# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/engyelgamal18/flyrank-ml-internship-engy/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content item for one client on one report date. For this assignment i will use a mid panel month, March 2026, so the analysis window is 01-03-2026 to 31-03-2026. This avoids using the final month of the daraset as a development window. i will use the fact_content_daily_performance table for this lane

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: impressions, clicks, average position, CTR and engagement related signals that are avaliable before decision moment.
Label/proxy: whether the content later declines or needs refresh attention
Context: content_id,client_id and report_date used for identifying, grouping and splitting rows but not as model features.
Excluded: any label derived or future information fields because they would leak the answer into the model.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [20]:
from huggingface_hub import whoami

print(whoami(token=hf_token))

{'type': 'user', 'id': '6a91f6a10a91fbd28c0bb65a', 'name': 'Engyelgamal', 'fullname': 'Engy Fouad Ahmed Mohamed Elgamal', 'email': 'engyelgamal18@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/25621e55311dd7ad4267976744a3afdd.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'flyrank-colab', 'role': 'read', 'createdAt': '2026-08-28T21:17:04.339Z'}}}


In [21]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
print("HF token loaded:",hf_token is not None)

HF token loaded: True


In [22]:
import os
os.environ["HF_TOKEN"] = hf_token
print("Ready to access FlyRank warehouse")

Ready to access FlyRank warehouse


In [23]:
from huggingface_hub import list_repo_files
files = list_repo_files(
    "FlyRank/internship-warehouse",
    repo_type="dataset",
    token=hf_token
)

march_files = [
    f for f in files
    if "fact_content_daily_performance" in f
    and "2026-03" in f
]

print("March files found:", len(march_files))
march_files[:10]

March files found: 1


['fact_content_daily_performance/month=2026-03/data_0.parquet']

In [24]:
from huggingface_hub import hf_hub_download
march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=march_files[0],
    repo_type="dataset",
    token=hf_token
)

print("March file ready:", march_path)

March file ready: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [25]:
import pandas as pd

march_df = pd.read_parquet(march_path)

grain_check = (
    march_df.groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
)

print("Duplicate grain rows:", len(grain_check))
grain_check.head()

Duplicate grain rows: 0


,report_date,client_hash_id,content_hash_id,count


In [26]:
print("Row count:", len(march_df))
print("Date range:", march_df["report_date"].min(), "to", march_df["report_date"].max())

Row count: 9841378
Date range: 2026-03-01 to 2026-03-31


In [27]:
import duckdb

availability_check = duckdb.query("""
Select count(*) AS avaliable_rows
From march_df
WHERE ga4_data_available IS TRUE
""").df()

availability_check

,avaliable_rows
0,413966


In [28]:
print(march_df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [29]:
feature_frame = march_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_total_engagement_sec"
    ]
].copy()

feature_frame.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_total_engagement_sec
0,20,0,3.350000,NaN,NaN
1,1,0,0.000000,NaN,NaN
2,125,1,4.928000,NaN,NaN
3,7,0,4.000000,NaN,NaN
4,11,0,2.272727,NaN,NaN


gsc_impressions: knowable at the decision moment because it is historical Search Console visibility data already observed before ranking pages.
gsc_clicks: knowable at the decision moment because it records clicks that have already happened.
gsc_avg_position: knowable at the decision moment because it summarizes the page's observed search position before the decision.
ga4_pageviews: knowable at the decision moment when ga4_data_available IS TRUE because it is historical traffic already observed.
ga4_total_engagement_sec: knowable at the decision moment when ga4_data_available IS TRUE because it measures engagement that has already occured.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One limitation of this slice is that GA4 is not avaliable for every row. In March 2026, only rows with ga4_data_available IS TRUE should be used for GA4 based features. This means the analysis may exclude some content and clients so the results may not represent the full dataset equally.

In [30]:
[c for c in march_df.columns if "trend"in c.lower() or "decline" in c.lower()]

[]

In [31]:
leak_df = march_df[
    ["report_date", "client_hash_id", "content_hash_id", "gsc_clicks"]
].copy()

leak_df = leak_df.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
)

leak_df["next_day_clicks"] = (
    leak_df
    .groupby(["client_hash_id", "content_hash_id"])["gsc_clicks"]
    .shift(-1)
)

leak_df["declined_next_day"] =(
    leak_df["next_day_clicks"] < leak_df["gsc_clicks"]
).astype(int)

leak_df[
    ["gsc_clicks", "next_day_clicks", "declined_next_day"]
].head()


,gsc_clicks,next_day_clicks,declined_next_day
363151,0,0.0,0
778019,0,0.0,0
87689,0,0.0,0
1472261,0,0.0,0
899536,0,0.0,0


In [32]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model_df = leak_df.dropna(subset=["next_day_clicks"]).sample(
    n=100000,
    random_state=42
)

y=model_df["declined_next_day"]

train_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.2,
    random_state=42
)

#Honest model: only information avaliable today
honest_model = LogisticRegression(max_iter=1000)
honest_model.fit(
    model_df.loc[train_idx,["gsc_clicks"]],
    y.loc[train_idx]
)

honest_score = accuracy_score(
    y.loc[test_idx],
    honest_model.predict(model_df.loc[test_idx, ["gsc_clicks"]])
)

#Leaky model: includes tomorrow' clicks
leaky_model = LogisticRegression(max_iter=1000)
leaky_model.fit(
    model_df.loc[train_idx,["gsc_clicks", "next_day_clicks"]],
    y.loc[train_idx]
)

leaky_score = accuracy_score(
    y.loc[test_idx],
    leaky_model.predict(
        model_df.loc[test_idx,["gsc_clicks", "next_day_clicks"]]
    )
)

print("Honest accuracy:", round(honest_score, 3))
print("Leaky accuracy:", round(leaky_score, 3))

Honest accuracy: 0.978
Leaky accuracy: 1.0


The honest model achieved 0.978 accuracy, while the leaky model reached 1.000 after including next_day_clicks. This is not a real improvement because next_day_clicks comes from the future and helps define the label. I would remove this feature and keep the honest model result.

In [33]:
honest_features = ["gsc-clicks"]

print("Final fearures kept:,", honest_features)
print("Leaky feature removed:next_day_clicks")
print("Honest accuracy kept:", round(honest_score, 3))

Final fearures kept:, ['gsc-clicks']
Leaky feature removed:next_day_clicks
Honest accuracy kept: 0.978


## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.